In [ ]:
import pandas as pd
import glob

: 

In [ ]:
glob_path = "./data/*.csv"

In [ ]:
glob_path

In [ ]:
glob_list = glob.glob(glob_path)

In [ ]:
df = pd.read_csv(glob_list[0])

In [ ]:
# Data Cleaning

df.isnull().sum()

In [ ]:
df.isna().sum()

In [ ]:
df.size

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
# Revenue

# Total Revenue
# Revenue trend (daily/weekly/monthly)
# Average Order Value (AOV)

# Customers

# New vs Returning customers
# Customer Lifetime Value (basic version is fine)
# Top customers

# Products

# Top selling products
# Low-performing products
# Category performance

# Operations

# Order volume trends
# Refund / cancellation rate (if data exists)
# 📈 Dashboard Views

# Think like a founder who has 5 mins, not an analyst.

# Revenue trend line
# Top products bar chart
# Customer segmentation pie
# Orders over time
# Heatmap (optional advanced)

In [ ]:
# Revenue

# Total Revenue
# Revenue trend (daily/weekly/monthly)
# Average Order Value (AOV)

In [ ]:
df.head()

In [ ]:
total_revenue = df['total_price'].sum()

In [ ]:
print(f"${(total_revenue)/1000000:.2f}M")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#### Date filters

df['date'] = pd.to_datetime(df['date'])

In [ ]:
class DateFilter:
    def __init__(self, df, filter_type=None, filter_data=None, token=None):
        self.df = df
        self.filter_type = filter_type
        self.filter_data = filter_data
        self.token = token

    def apply(self):
        
        if not self.filter_type or not self.filter_data or not self.token:
            return self.df  # nothing to apply

        # map filter type → pandas datetime accessor
        attr_map = {
            "year": "year",
            "month": "month",
            "day": "day"
        }

        attr = attr_map.get(self.filter_type)

        if not attr:
            return self.df

        series = getattr(self.df['date'].dt, attr)

        if self.token == "between":
            start, end = self.filter_data
            return self.df[(series >= start) & (series <= end)]

        elif self.token == "more":
            return self.df[series >= self.filter_data[0]]

        elif self.token == "less":
            return self.df[series <= self.filter_data[0]]

        return self.df

In [ ]:
filters = [
    {"type": "year", "data": [2009], "token": "more"},
    {"type": "month", "data": [1, 6], "token": "between"},
    {"type": "day", "data": [1, 15], "token": "between"},
]

In [ ]:
df.size

In [ ]:
df.head()

In [ ]:
df['Country'].unique()

In [ ]:
country = ['Canada', 'Germany']
def country_filter(df, country):
    if not country:
        return df
    return df[df['Country'].isin(country)]

In [ ]:
product = []

def product_filter(df, product):
    if not product:
        return df
    return df[df['Description'].isin(product)]

In [ ]:
df['Description'].value_counts()

In [ ]:
# Apply filters

filters = [
    {"type" : "product", "data" : ['WHITE HANGING HEART T-LIGHT HOLDER', 'REGENCY CAKESTAND 3 TIER', 'PARTY BUNTING', 'ASSORTED COLOUR BIRD ORNAMENT', 'STRAWBERRY CERAMIC TRINKET BOX']},
    {"type" : "country", "data" : ['Canada', 'Germany']},
    {"type": "year", "data": [2009], "token": "more"},
    {"type": "month", "data": [1, 6], "token": "between"},
    {"type": "day", "data": [1, 15], "token": "between"},
]

In [ ]:
filter_map = {
    "product" : product_filter,
    "country" : country_filter,
}

In [ ]:
def apply_filters(df, filters):
    df = df.copy()  # avoid modifying original df
    df['date'] = pd.to_datetime(df['date'])  # ensure date is datetime
    for i in filters:
        type =i['type']
        data = i['data']

        if type in filter_map:
            filter_func = filter_map[type]
            df = filter_func(df, data)
            
        elif type in ["year", "month", "day"]:
            token = i['token']
            date_filter = DateFilter(df, filter_type=type, filter_data=data, token=token)
            df = date_filter.apply()

    return df

In [ ]:
updated_df = apply_filters(df, filters)

In [ ]:
updated_df.head()

In [ ]:
ui = input("enter list")
ui = ui.replace(" ","")
ui = ui.split(",")

In [ ]:
sns.lineplot(data = updated_df, x = updated_df['date'].dt.year, y="total_price")

In [ ]:
def date_wise_trends(df, date_part, on = None, metric = None):
    attr_map = {
        "year": "year",
        "month": "month",
        "day": "day"
    }

    attr = attr_map.get(date_part)

    # Extract the date part properly
    df[date_part] = getattr(df['date'].dt, attr)

    year_group = (
        df.groupby(date_part)
        .agg({on : metric})
        .reset_index()
        .sort_values(date_part, ascending=True)
    )

    year_group[date_part] = year_group[date_part].astype(str)



    sns.lineplot(data=year_group, x=date_part, y=on)

In [ ]:
date_wise_trends(df, "month")

In [ ]:
## Average Order Value (AOV)
aov = f"{(total_revenue / df['Invoice'].nunique()):.2f}"

In [ ]:
aov

In [ ]:
#Customers
# 1. New Vs Returning

cust_seg = df.groupby('cust_id').agg({
    "date":"nunique"
}).reset_index().sort_values("date", ascending=False)


cust_seg['segment'] = cust_seg['date'].apply(lambda x: 'New' if x == 1 else 'Returning')

new_cust = (cust_seg[cust_seg['segment'] == 'New'].shape[0] / cust_seg.shape[0]) * 100
returning_cust = (cust_seg[cust_seg['segment'] == 'Returning'].shape[0] / cust_seg.shape[0]) * 100


In [ ]:
new_cust

In [ ]:
returning_cust

In [ ]:
# Customer Lifetime Value

# CLV = purchase_frequency * average_order_value * customer_lifespan

# customer_lifespan = 1/churn_rate


# churn_rate = (customer_lost/ customer_at_beginning_of_period)


### Churn Rate Calculation

year = 2010

filters = [
    {"type": "year", "data": [2010], "token": "less"}
]

for i in filters:
    type =i['type']
    data = i['data']
    token = i['token']

    date_filter = DateFilter(df, filter_type=type, filter_data=data, token=token)
    req_cust = date_filter.apply()

customer_after_period = df[df['date'].dt.year > year]

cust_before = set(req_cust['cust_id'].unique())
cust_after = set(customer_after_period['cust_id'].unique())


customer_lost = len(cust_before - cust_after)
churn_rate = (customer_lost / len(cust_before))
average_lifespan = 1/(churn_rate)



In [ ]:
average_lifespan

In [ ]:
purchase_frequency = df.groupby('cust_id')['Invoice'].nunique().mean()

In [ ]:
purchase_frequency

In [ ]:
customer_life_time_value = float(f"{(purchase_frequency * float(aov) * average_lifespan):.2f}")

In [ ]:
customer_life_time_value

In [ ]:
#Top / Bottom N Customers

top_or_bottom = None
n = None
on = None
metric = None

def top_bottom_customers(df, top_or_bottom = None, n = None, on = None, metric = None):
    if not top_or_bottom or not n or not on or not metric:
        return df
    
    cust_revenue = df.groupby(on).agg({
        metric : "sum"
    }).sort_values(metric, ascending = False)

    if top_or_bottom == "top":
        return cust_revenue.head(n)
    
    elif top_or_bottom == "bottom":
        return cust_revenue.tail(n)

In [ ]:
top_bottom_customers(df, top_or_bottom="top", n = 10, on = "cust_id", metric = "total_price")

In [ ]:
#Refund rate

def refund_rate(df):
    total_orders= df['Invoice'].nunique()
    refunded_orders = df[df['Quantity']<0]['Invoice'].nunique()
    return float(f"{((refunded_orders/total_orders) * 100):.2f}")

In [ ]:
refund_rate(df)

In [ ]:
#orders KPI
total_orders = df['Invoice'].nunique()
total_units_sold = df[df['Quantity']>0]['Quantity'].sum()



In [ ]:
total_units_sold

In [ ]:
#orders over time 
date_wise_trends(df, "day", on = "Invoice", metric = "nunique")

In [2]:
import pandas as pd

df = pd.read_csv("./data/retail_sales_dataset.csv")
df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


In [3]:
def add_age_groups(df):
    if 'Age' not in df.columns:
        return df
    
    df['age_group'] = [
        "early earners" if age >=18 and age <=24 
        else "Young professionals" if age >=25 and age <=34 
        else "Stable income" if age >=35 and age <= 44 
        else "Senior citizens" 
        for age in df['Age']
    ]
    return df

In [ ]:
def metric_wise_average_order_value(df,metric):
    if metric not in df.columns:
        return df
    
    metric_aov = df.groupby(metric).agg(
      total_revenue = ('total_price', 'sum'),
      total_orders = ('Invoice', 'nunique')
   ).reset_index()
    
    metric_aov['average_order_value'] = metric_aov['total_revenue'] / metric_aov['total_orders']
    return metric_aov[[metric, 'average_order_value']]


def revenue_contib_by_metric(df, metric):
    if metric not in df.columns:
        return df
    
    contribution = df.groupby(metric).agg(
        total_revenue = ('total_price', 'sum')
    ).reset_index()

    total_revenue = contribution['total_revenue'].sum()
    contribution['revenue_contribution'] = (contribution['total_revenue'] / total_revenue) * 100

    return contribution[[metric, 'revenue_contribution']]



    
